# 이커머스 상품 누끼 자동화 PoC — Colab 실행

위에서부터 차례로 실행하면 됩니다. GPU는 없어도 되지만, 있으면 SAM이 빨라집니다.
(메뉴 → 런타임 → 런타임 유형 변경 → T4 GPU)

전체 12장 처리에 **약 2~3분** 걸립니다. 대부분은 모델 가중치를 내려받는 시간입니다.

문제 정의서와 검증 결과는 저장소의 `docs/` 폴더에 있습니다.

## 1. 저장소 내려받기

In [ ]:
!git clone -q https://github.com/songyee-ai/ecommerce-cutout-poc.git
%cd ecommerce-cutout-poc
!ls data/samples

## 2. 패키지 설치

Colab에는 `torch`와 `transformers`가 미리 깔려 있습니다. 누끼 전용 모델을 쓰는 `rembg`만 추가로 설치합니다.

In [ ]:
!pip install -q rembg onnxruntime

## 3. 실행 환경과 라이선스 확인

라이선스는 저장소 첫 화면 표시가 아니라 **설치한 패키지의 배포 정보**에서 확인하는 편이 정확합니다.
한 저장소 안에서 코드와 가중치의 조건이 다르게 붙어 있기도 합니다.

In [ ]:
!python poc/check_env.py

## 4. 후보 모델 실행

**후보를 한 번에 돌리지 않습니다.** `--backends sam,u2net` 처럼 붙여 쓰면 두 모델이 동시에
메모리에 올라가 무료 Colab 한도(약 13GB)를 넘기고 프로세스가 강제 종료됩니다.
실제로 겪은 문제라, 후보마다 프로세스를 나눕니다.

In [ ]:
RUN_ID = "my_run"

# 후보 1 — SAM (범용 분할 모델, 사진 중앙에 점 1개를 찍어 지목)
!python poc/run_cutout.py --images data/samples --out outputs --backends sam --run-id $RUN_ID

In [ ]:
# 후보 2 — U2-Net (누끼 전용 모델, 지목 불필요)
!python poc/run_cutout.py --images data/samples --out outputs --backends u2net --run-id $RUN_ID

In [ ]:
# 후보별 결과를 하나로 합친다
!python poc/run_cutout.py --images data/samples --out outputs --run-id $RUN_ID --merge-only

## 5. 점수 보기

점수는 반올림하지 않고 그대로 남깁니다. 임계값을 정할 때 씁니다.

- `sam_iou` — SAM이 스스로 매긴 마스크 품질 예측
- `sam_stability` — 경계를 흔들어도 면적이 유지되는 정도
- `coverage` — 상품으로 판정된 픽셀 비율
- `edge_ambiguity` — 알파가 중간값인 픽셀 비율 (경계가 애매한 정도)
- `touches_border` — 마스크가 사진 테두리에 닿은 비율

In [ ]:
import pandas as pd

df = pd.read_csv(f"outputs/{RUN_ID}/scores.csv")
display(df)
print(df.groupby("backend")["seconds"].agg(["sum", "mean"]))

## 6. 결과를 눈으로 확인

숫자만 보고 판단하면 **값이 의도한 것을 재고 있지 않은 경우**를 놓칩니다.
한 장에 세 칸이 붙어 있습니다 — `원본 | 마스크(흰=상품) | 체커보드 위 합성`.
세 번째 칸의 회색 체크무늬가 투명하게 뚫린 부분입니다.

In [ ]:
from pathlib import Path
from IPython.display import display, Image

for backend in ["sam", "u2net"]:
    print(f"\n{'=' * 30}  {backend}  {'=' * 30}")
    for p in sorted(Path(f"outputs/{RUN_ID}/{backend}/check").glob("*.jpg")):
        print(p.stem)
        display(Image(filename=str(p), width=900))

## 7. 채점

`outputs/<RUN_ID>/scoring.csv` 의 `pass_fail` 칸을 사람이 직접 채웁니다.
판정 기준(R1~R3)은 `docs/01_problem_definition.md` 4절에 있고, **모델을 돌리기 전에** 정해져 있습니다.

결과를 본 뒤에 기준을 바꾸면 채점이 성립하지 않습니다.

이 PoC에서 나온 판정과 실패 사유는 `outputs/run2/scoring.csv` 에 있습니다.